# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following best practices for referencing all dataset elements by their `@id`. You will:

- Load Croissant metadata
- Explore available record sets and fields
- Extract and analyze tabular record data
- Perform basic EDA and visualize key attributes

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print human-readable metadata
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review record sets (`@id`s), fields, and columns available in the dataset. All IDs are referenced by their explicit Croissant `@id`.

In [ ]:
from pprint import pprint

# List all record sets by their @id
print("Available record sets (@id):")
record_sets = list(metadata.recordSet)
for rs in record_sets:
    print(f"- {rs['@id']}")

# For each record set, print details of fields and columns by @id
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    if 'field' in rs:
        print("Fields:")
        for field in rs['field']:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"  - {field_id}")
    if 'column' in rs:
        print("Columns:")
        for column in rs['column']:
            col_id = column['@id'] if isinstance(column, dict) and '@id' in column else column
            print(f"  - {col_id}")

## 3. Data Extraction
Load each record set into a DataFrame for analysis. All record, field, and column IDs are referenced by their `@id` for full traceability.

> **Tip:** If you do not know the exact `@id`s, consult the previous overview or the dataset's Croissant file.

In [ ]:
# Collect all record set @ids (as list of strings)
record_set_ids = [rs['@id'] for rs in record_sets]

# Create a dict of DataFrames keyed by record set @id
dataframes = {}
for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(list(records_iter))
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set: {record_set_id} | Rows: {df.shape[0]}, Columns: {df.shape[1]}")

if dataframes:
    # Just pick the first for exploration:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns for record set '{main_record_set_id}' (@id):")
    print(list(dataframes[main_record_set_id].columns))
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets with tabular data found.")

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA steps. Examples: filtering by a numeric field, normalization, and grouping. All columns are referenced by their Croissant `@id`.

> **Note:** Adjust the `numeric_field_id` and `group_field_id` variables below to match actual field/column `@id`s from your dataset record sets.

In [ ]:
# EDA example -- update these IDs after inspecting columns above
# Example: choosing numeric field and grouping field by their @id

record_set_id = main_record_set_id  # Use the first (or main) record set
df = dataframes[record_set_id]

# Find candidate numeric fields (columns with float/int dtype)
numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected (by `@id`):", numeric_columns)
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    # Try to find a grouping field that is categorical/non-numeric
    group_field_candidates = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
    group_field_id = group_field_candidates[0] if group_field_candidates else None

    # Filtering example: values > threshold
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a key, if available
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Let's visualize the distribution of a numeric field and (optionally) compare means across groups. All column references use `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_columns:
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    # Box plot by group field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.grid(axis='y')
        plt.tight_layout()
        plt.show()
else:
    print("Cannot visualize: missing numeric field or no data.")

## 6. Conclusion

- Demonstrated programmatic loading and overview of a Croissant dataset with `mlcroissant`, referencing all data entities by `@id`.
- Showed how to load and view the available record sets, fields, and columns.
- Extracted record set data into Pandas DataFrames for EDA, applying numeric filtering, normalization, and grouping by field.
- Included example visualizations of numeric distributions and group comparisons.

**Next steps:** Apply domain-specific analysis to the record sets and fields, map data to the dataset documentation, and use the ID-based workflow for reproducible data processing.

For more examples using Croissant/FAIR datasets, see the [mlcroissant documentation](https://mlcroissant.readthedocs.io/).